# 03 · Evaluate on the held-out test regions

The test regions (Mannheim, Lyon, Picardy, Thuringian Forest, Lake Balaton, Lusatia mines) were never seen during training or model selection. Each region is predicted **as a whole 768 × 768 scene** for each of the 12 months, exactly as the prototype does.

Reported:
1. Overall accuracy, mIoU, per-class IoU, precision, recall, F1
2. Confusion matrix
3. Accuracy **per month** (how much seasonality hurts a radar-only model)
4. **Temporal stability**: how often a pixel changes class between consecutive months, although the reference label does not change. This is the noise floor of the prototype's month-to-month change figures.
5. Qualitative maps

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd() if Path("config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch

import config as C
from train_utils import load_regions, metrics_from_cm
from lulc_model import load_checkpoint, predict, temporal_majority

DATA_DIR, MODEL_DIR = ROOT / "data", ROOT / "models"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model, ckpt = load_checkpoint(MODEL_DIR / "best_model.pth", DEVICE)
MEAN, STD = ckpt["mean"], ckpt["std"]
print(f"Loaded checkpoint from epoch {ckpt.get('epoch')} (val mIoU {ckpt.get('val_mIoU', float('nan')):.3f})")
cmap = ListedColormap(C.CLASS_COLORS)

## 1 · Predict all test regions × 12 months

In [ ]:
test_regions = load_regions(DATA_DIR, "test")
K = C.NUM_CLASSES

def cm_np(pred, target):
    m = (target != C.IGNORE_INDEX) & (pred != C.IGNORE_INDEX)
    return np.bincount(target[m].astype(int) * K + pred[m], minlength=K * K).reshape(K, K)

preds = {}          # name -> (12, H, W)
cm_total = np.zeros((K, K), dtype=np.int64)
cm_month = np.zeros((12, K, K), dtype=np.int64)
cm_smooth = np.zeros((K, K), dtype=np.int64)
for r in test_regions:
    maps = np.stack([predict(model, r["s1"][t].astype(np.float32), MEAN, STD, DEVICE) for t in range(12)])
    preds[r["name"]] = maps
    for t in range(12):
        c = cm_np(maps[t], r["label"])
        cm_month[t] += c; cm_total += c
    sm = temporal_majority(maps, 3)
    for t in range(12):
        cm_smooth += cm_np(sm[t], r["label"])
    print(f"{r['name']:20s} done")

## 2 · Overall metrics

In [ ]:
m = metrics_from_cm(cm_total)
m_s = metrics_from_cm(cm_smooth)
print(f"Overall accuracy : {m['overall_accuracy']:.3f}")
print(f"mIoU             : {m['mIoU']:.3f}")
print(f"mean F1          : {m['mean_F1']:.3f}")
print(f"mIoU with 3-month majority smoothing: {m_s['mIoU']:.3f}")
per_class = pd.DataFrame({k: m[k] for k in ["IoU", "precision", "recall", "F1"]}, index=C.CLASS_NAMES)
per_class["support_px"] = cm_total.sum(1)
per_class.round(3)

In [ ]:
cm_norm = cm_total / np.maximum(cm_total.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
for i in range(K):
    for j in range(K):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm_norm[i, j] > .5 else "black", fontsize=9)
ax.set_xticks(range(K), C.CLASS_NAMES, rotation=35, ha="right"); ax.set_yticks(range(K), C.CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("WorldCover (reference)")
ax.set_title("Test confusion matrix (row-normalised = recall)")
plt.colorbar(im, fraction=.046); plt.tight_layout()
plt.savefig(MODEL_DIR / "test_confusion_matrix.png", dpi=150); plt.show()

## 3 · Performance per month

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pm = [metrics_from_cm(cm_month[t]) for t in range(12)]
per_month = pd.DataFrame({"mIoU": [x["mIoU"] for x in pm], "OA": [x["overall_accuracy"] for x in pm],
                          **{f"IoU {n}": [x["IoU"][k] for x in pm] for k, n in enumerate(C.CLASS_NAMES)}},
                         index=month_names)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(month_names, per_month["mIoU"], "k-o", lw=2.5, label="mIoU")
for k, (n, col) in enumerate(zip(C.CLASS_NAMES, C.CLASS_COLORS)):
    ax.plot(month_names, per_month[f"IoU {n}"], "-", color=col, alpha=.8, label=n)
ax.set_ylabel("IoU"); ax.set_title("Test IoU by month of 2021"); ax.grid(alpha=.3)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout()
plt.savefig(MODEL_DIR / "test_iou_by_month.png", dpi=150); plt.show()
per_month.round(3)

## 4 · Temporal stability
The reference label is constant over the year, so every month-to-month class change predicted on these regions is a **false change**. The share of pixels that flip between consecutive months is therefore the noise level to keep in mind when reading the prototype's change figures.

In [ ]:
rows = []
for name, maps in preds.items():
    flips = []
    for t in range(11):
        ok = (maps[t] != C.IGNORE_INDEX) & (maps[t + 1] != C.IGNORE_INDEX)
        flips.append(100 * (maps[t][ok] != maps[t + 1][ok]).mean())
    sm = temporal_majority(maps, 3)
    flips_s = []
    for t in range(11):
        ok = (sm[t] != C.IGNORE_INDEX) & (sm[t + 1] != C.IGNORE_INDEX)
        flips_s.append(100 * (sm[t][ok] != sm[t + 1][ok]).mean())
    rows.append({"region": name, "pixels changing class per month (%)": np.mean(flips),
                 "with 3-month smoothing (%)": np.mean(flips_s)})
stab = pd.DataFrame(rows).set_index("region")
stab.loc["mean"] = stab.mean()
stab.round(2)

## 5 · Qualitative results

In [ ]:
def show_region(name, months=(0, 3, 6, 9)):
    r = next(x for x in test_regions if x["name"] == name)
    maps = preds[name]
    fig, axes = plt.subplots(1, len(months) + 1, figsize=(4 * (len(months) + 1), 4.3))
    axes[0].imshow(np.ma.masked_equal(r["label"], 255), cmap=cmap, vmin=0, vmax=K - 1, interpolation="nearest")
    axes[0].set_title(f"{name}: WorldCover 2021")
    for a, t in zip(axes[1:], months):
        a.imshow(np.ma.masked_equal(maps[t], 255), cmap=cmap, vmin=0, vmax=K - 1, interpolation="nearest")
        a.set_title(f"Prediction {month_names[t]}")
    for a in axes: a.axis("off")
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in C.CLASS_COLORS]
    fig.legend(handles, C.CLASS_NAMES, loc="lower center", ncol=K)
    plt.tight_layout(rect=(0, .06, 1, 1)); plt.show()

for name in preds:
    show_region(name)

## 6 · Save metrics

In [ ]:
out = {"test_overall": m, "test_overall_smoothed": m_s,
       "per_month_mIoU": per_month["mIoU"].round(4).tolist(),
       "temporal_flip_rate_pct": stab.loc["mean"].round(3).to_dict(),
       "confusion_matrix": cm_total.tolist(), "class_names": C.CLASS_NAMES}
(MODEL_DIR / "test_metrics.json").write_text(json.dumps(out, indent=2))
print("Saved", MODEL_DIR / "test_metrics.json")